In [6]:
%pip install transformers

  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 17.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 14.3 MB/s  0:00:00 eta 0:00:01
Using cached annotated_doc-0.0.4-py3-none-any.whl (5.3 kB)
Using cached mdurl-0.1.2-py3-none-any.whl (10.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [transformers] [transformers]y]
Note: you may need to restart the kernel to use updated packages.


In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [ ]:
tokenizer

In [ ]:
tokenizer.encode("sentence")

In [ ]:
tokenizer.decode([11111])

# Dataset-specific workflow: FLORES-101

Everything below belongs to the dataset experiment. FLORES-101 replaces the earlier OpenWebText corpus; the introductory tokenizer examples above remain separate and unchanged.

## Dataset setup and corpus loading

In [ ]:
%pip install "datasets<4" evaluate "transformers[sentencepiece]" pandas matplotlib

In [ ]:
from datasets import load_dataset

# FLORES-101 translations are aligned by row. These ten languages cover
# Latin, Cyrillic, and Indic writing systems.
SELECTED_LANGUAGES = {
    "eng": {"language": "English", "script": "Latin"},
    "fra": {"language": "French", "script": "Latin"},
    "deu": {"language": "German", "script": "Latin"},
    "spa": {"language": "Spanish", "script": "Latin"},
    "rus": {"language": "Russian", "script": "Cyrillic"},
    "bul": {"language": "Bulgarian", "script": "Cyrillic"},
    "ukr": {"language": "Ukrainian", "script": "Cyrillic"},
    "hin": {"language": "Hindi", "script": "Indic (Devanagari)"},
    "ben": {"language": "Bengali", "script": "Indic (Bengali)"},
    "mar": {"language": "Marathi", "script": "Indic (Devanagari)"},
}

# The repository uses a dataset loading script, so datasets<4 and
# trust_remote_code=True are required. The 'all' config downloads once and
# exposes every aligned translation as a sentence_<language code> column.
dataset = load_dataset(
    "gsarti/flores_101",
    "all",
    split="dev",
    trust_remote_code=True,
)

# Use the full dev split for training so the corpus has enough pair diversity
# to support 10k-30k merges. Keep the analysis sample fixed at 100 aligned rows.
TRAIN_SENTENCES_PER_LANGUAGE = len(dataset)
ANALYSIS_SENTENCES_PER_LANGUAGE = 100
corpus = [
    sentence
    for code in SELECTED_LANGUAGES
    for sentence in dataset[f"sentence_{code}"][:TRAIN_SENTENCES_PER_LANGUAGE]
]

print(f"Training corpus: {len(corpus)} sentences across {len(SELECTED_LANGUAGES)} languages")

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

## BPE training on the multilingual corpus

In [ ]:
from collections import defaultdict

word_freqs = defaultdict(int)

for text in corpus:
    words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
    new_words = [word for word, offset in words_with_offsets]
    for word in new_words:
        word_freqs[word] += 1

print(f"Unique pre-tokens: {len(word_freqs)}")
print(list(word_freqs.items())[:20])

In [ ]:
alphabet = []

for word in word_freqs.keys():
    for letter in word:
        if letter not in alphabet:
            alphabet.append(letter)
alphabet.sort()

print(f"Alphabet size: {len(alphabet)}")
print(alphabet)

In [ ]:
vocab = ["<|endoftext|>"] + alphabet.copy()

In [ ]:
splits = {word: [c for c in word] for word in word_freqs.keys()}

In [ ]:
def compute_pair_freqs(splits):
    pair_freqs = defaultdict(int)
    for word, freq in word_freqs.items():
        split = splits[word]
        if len(split) == 1:
            continue
        for i in range(len(split) - 1):
            pair = (split[i], split[i + 1])
            pair_freqs[pair] += freq
    return pair_freqs

In [ ]:
pair_freqs = compute_pair_freqs(splits)

for i, key in enumerate(pair_freqs.keys()):
    print(f"{key}: {pair_freqs[key]}")
    if i >= 5:
        break

In [ ]:
best_pair = ""
max_freq = None

for pair, freq in pair_freqs.items():
    if max_freq is None or max_freq < freq:
        best_pair = pair
        max_freq = freq

print(best_pair, max_freq)

In [ ]:
# Learn every merge from the multilingual corpus instead of seeding an
# English-specific merge.
merges = {}

In [ ]:
def merge_pair(a, b, splits):
    for word in word_freqs:
        split = splits[word]
        if len(split) == 1:
            continue

        i = 0
        while i < len(split) - 1:
            if split[i] == a and split[i + 1] == b:
                split = split[:i] + [a + b] + split[i + 2 :]
            else:
                i += 1
        splits[word] = split
    return splits

In [ ]:
print(f"Initial alphabet size: {len(alphabet)}")
print(f"Most frequent initial pair: {best_pair} ({max_freq} occurrences)")

In [ ]:
import heapq
from time import perf_counter

# Practical BPE experiments commonly use vocabularies with tens of thousands
# of learned merges. Train once to 30k and save nested checkpoints so all
# settings use the exact same merge-learning trajectory.
MERGE_CHECKPOINTS = (10_000, 20_000, 30_000)

def train_bpe_checkpoints(initial_splits, word_freqs, checkpoints):
    trained_splits = {word: pieces.copy() for word, pieces in initial_splits.items()}
    pair_freqs = defaultdict(int)
    pair_to_words = defaultdict(set)

    for word, pieces in trained_splits.items():
        adjacent_pairs = list(zip(pieces, pieces[1:]))
        for pair in adjacent_pairs:
            pair_freqs[pair] += word_freqs[word]
        for pair in set(adjacent_pairs):
            pair_to_words[pair].add(word)

    heap = [(-frequency, pair) for pair, frequency in pair_freqs.items() if frequency > 0]
    heapq.heapify(heap)
    learned_merges = {}
    checkpoints_out = {}
    target = max(checkpoints)

    while len(learned_merges) < target and heap:
        while heap:
            negative_frequency, best_pair = heapq.heappop(heap)
            if pair_freqs.get(best_pair, 0) == -negative_frequency and negative_frequency < 0:
                break
        else:
            break

        a, b = best_pair
        merged_token = a + b
        affected_words = list(pair_to_words.get(best_pair, ()))
        changed_pairs = set()

        for word in affected_words:
            old_pieces = trained_splits[word]
            old_pairs = list(zip(old_pieces, old_pieces[1:]))
            for pair in set(old_pairs):
                pair_to_words[pair].discard(word)
            for pair in old_pairs:
                pair_freqs[pair] -= word_freqs[word]
                changed_pairs.add(pair)

            new_pieces = []
            index = 0
            while index < len(old_pieces):
                if index < len(old_pieces) - 1 and old_pieces[index] == a and old_pieces[index + 1] == b:
                    new_pieces.append(merged_token)
                    index += 2
                else:
                    new_pieces.append(old_pieces[index])
                    index += 1
            trained_splits[word] = new_pieces

            new_pairs = list(zip(new_pieces, new_pieces[1:]))
            for pair in new_pairs:
                pair_freqs[pair] += word_freqs[word]
                changed_pairs.add(pair)
            for pair in set(new_pairs):
                pair_to_words[pair].add(word)

        learned_merges[best_pair] = merged_token
        for pair in changed_pairs:
            frequency = pair_freqs.get(pair, 0)
            if frequency > 0:
                heapq.heappush(heap, (-frequency, pair))
            else:
                pair_freqs.pop(pair, None)
                pair_to_words.pop(pair, None)

        merge_count = len(learned_merges)
        if merge_count in checkpoints:
            checkpoints_out[merge_count] = learned_merges.copy()
            print(f"Saved {merge_count:,}-merge checkpoint")
        if merge_count % 1_000 == 0:
            # Rebuild periodically to discard stale heap entries.
            heap = [(-frequency, pair) for pair, frequency in pair_freqs.items() if frequency > 0]
            heapq.heapify(heap)

    return trained_splits, learned_merges, checkpoints_out

training_start = perf_counter()
splits, merges, merge_checkpoints = train_bpe_checkpoints(
    splits, word_freqs, MERGE_CHECKPOINTS
)
training_seconds = perf_counter() - training_start
available_checkpoints = sorted(merge_checkpoints)
if not available_checkpoints:
    raise RuntimeError("The corpus did not contain enough mergeable pairs for the requested checkpoints.")

# Keep the largest checkpoint as the default tokenizer while retaining all
# checkpoints for the comparative analysis below.
merges = merge_checkpoints[available_checkpoints[-1]]
vocab = ["<|endoftext|>"] + alphabet.copy() + list(merges.values())
merge_ranks_by_size = {
    merge_count: {pair: rank for rank, pair in enumerate(rules)}
    for merge_count, rules in merge_checkpoints.items()
}
print(f"Training completed in {training_seconds:.1f} seconds")

In [ ]:
print(f"Requested checkpoints: {[f'{value:,}' for value in MERGE_CHECKPOINTS]}")
print(f"Available checkpoints: {[f'{value:,}' for value in available_checkpoints]}")
print(f"Largest learned merge table: {len(merges):,} rules")
print(list(merges.items())[:20])

In [ ]:
print(f"Final vocabulary size: {len(vocab):,}")
print(vocab[:30])

In [ ]:
def tokenize(text, merge_ranks=None):
    if merge_ranks is None:
        merge_ranks = merge_ranks_by_size[available_checkpoints[-1]]

    pre_tokenize_result = tokenizer._tokenizer.pre_tokenizer.pre_tokenize_str(text)
    pre_tokenized_text = [word for word, offset in pre_tokenize_result]
    tokenized_words = []

    for word in pre_tokenized_text:
        pieces = list(word)
        while len(pieces) > 1:
            candidate_pairs = set(zip(pieces, pieces[1:]))
            best_pair = min(
                (pair for pair in candidate_pairs if pair in merge_ranks),
                key=merge_ranks.get,
                default=None,
            )
            if best_pair is None:
                break

            a, b = best_pair
            merged_pieces = []
            index = 0
            while index < len(pieces):
                if index < len(pieces) - 1 and pieces[index] == a and pieces[index + 1] == b:
                    merged_pieces.append(a + b)
                    index += 2
                else:
                    merged_pieces.append(pieces[index])
                    index += 1
            pieces = merged_pieces
        tokenized_words.extend(pieces)

    return tokenized_words

In [ ]:
{
    f"{merge_count:,} merges": tokenize(
        "this is a sample sentence", merge_ranks_by_size[merge_count]
    )
    for merge_count in available_checkpoints
}

## Token-length comparison

Each FLORES row contains translations of the same source sentence. The following cells apply the 10k, 20k, and 30k BPE checkpoints to the first 100 aligned rows. `custom_bpe_token_count` is the number of tokens produced at a particular merge budget. The pretrained GPT-2 count remains as a byte-level BPE baseline.

In [ ]:
import pandas as pd

comparison_rows = []
for sentence_index in range(ANALYSIS_SENTENCES_PER_LANGUAGE):
    row = dataset[sentence_index]
    for code, metadata in SELECTED_LANGUAGES.items():
        sentence = row[f"sentence_{code}"]
        gpt2_token_count = len(tokenizer.encode(sentence, add_special_tokens=False))
        for merge_count in available_checkpoints:
            custom_tokens = tokenize(sentence, merge_ranks_by_size[merge_count])
            comparison_rows.append({
                "sentence_id": row["id"],
                "num_merges": merge_count,
                "language_code": code,
                "language": metadata["language"],
                "script": metadata["script"],
                "sentence": sentence,
                "character_count": len(sentence),
                "custom_bpe_token_count": len(custom_tokens),
                "gpt2_token_count": gpt2_token_count,
            })

lengths_df = pd.DataFrame(comparison_rows)
print(f"Computed {len(lengths_df):,} checkpoint-language-sentence measurements.")

In [ ]:
# Ten translations of one aligned sentence at every merge checkpoint.
lengths_df.loc[
    lengths_df["sentence_id"] == lengths_df["sentence_id"].min(),
    ["sentence_id", "num_merges", "language", "script", "sentence", "custom_bpe_token_count", "gpt2_token_count"],
].sort_values(["num_merges", "script", "language"])

In [ ]:
# Direct comparison for the first ten aligned IDs at all merge checkpoints.
first_ten_ids = sorted(lengths_df["sentence_id"].unique())[:10]
first_ten_custom_lengths = lengths_df[
    lengths_df["sentence_id"].isin(first_ten_ids)
].pivot(index=["num_merges", "sentence_id"], columns="language", values="custom_bpe_token_count")
first_ten_custom_lengths

In [ ]:
# Aggregate all 100 aligned sentences for every language and merge budget.
language_summary = (
    lengths_df.groupby(["num_merges", "script", "language"], as_index=False)
    .agg(
        mean_custom_bpe_tokens=("custom_bpe_token_count", "mean"),
        median_custom_bpe_tokens=("custom_bpe_token_count", "median"),
        mean_gpt2_tokens=("gpt2_token_count", "mean"),
        mean_characters=("character_count", "mean"),
    )
    .sort_values(["num_merges", "mean_custom_bpe_tokens"])
)
language_summary.round(2)

## Results visualizations

The first figure shows how mean sequence length changes from 10k to 30k merges. The second summarizes the full distribution at the largest checkpoint, and the third preserves FLORES alignment for ten sentence meanings at that checkpoint. Set `EXPORT_FIGURES = True` to save PDF and 300-DPI PNG versions.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

# Okabe-Ito colorblind-safe colors, kept consistent across all figures.
SCRIPT_COLORS = {
    "Latin": "#0072B2",
    "Cyrillic": "#D55E00",
    "Indic": "#009E73",
}

def script_family(script):
    return "Indic" if script.startswith("Indic") else script

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "axes.linewidth": 0.8,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

EXPORT_FIGURES = False

def finish_figure(fig, filename):
    fig.tight_layout()
    if EXPORT_FIGURES:
        fig.savefig(f"{filename}.pdf", bbox_inches="tight")
        fig.savefig(f"{filename}.png", bbox_inches="tight", dpi=300)
    plt.show()

In [ ]:
# Figure 1: shared-scale facets show the effect of increasing the merge budget.
language_to_family = {
    metadata["language"]: script_family(metadata["script"])
    for metadata in SELECTED_LANGUAGES.values()
}
MARKERS = ["o", "s", "^", "D"]
fig, axes = plt.subplots(1, 3, figsize=(10.2, 3.6), sharex=True, sharey=True)

for ax, family in zip(axes, ("Latin", "Cyrillic", "Indic")):
    family_languages = [
        language for language, language_family in language_to_family.items()
        if language_family == family
    ]
    for marker, language in zip(MARKERS, family_languages):
        plot_data = language_summary[language_summary["language"] == language]
        ax.plot(
            plot_data["num_merges"],
            plot_data["mean_custom_bpe_tokens"],
            color=SCRIPT_COLORS[family],
            marker=marker,
            linewidth=1.5,
            markersize=5,
            label=language,
        )
    ax.set_title(family, fontweight="bold")
    ax.set_xticks(available_checkpoints, [f"{value // 1000}k" for value in available_checkpoints])
    ax.set_xlabel("Learned merges")
    ax.grid(color="#D9D9D9", linewidth=0.6, alpha=0.8)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, loc="best")

axes[0].set_ylabel("Mean BPE sequence length (tokens)")
fig.suptitle("Token length decreases as the multilingual BPE vocabulary grows", fontweight="bold", y=1.02)
finish_figure(fig, "bpe_merge_budget_scaling")

In [ ]:
# Figure 2: full distributions at the largest, most practical checkpoint.
largest_checkpoint = available_checkpoints[-1]
largest_lengths = lengths_df[lengths_df["num_merges"] == largest_checkpoint]
language_order = (
    largest_lengths.groupby("language")["custom_bpe_token_count"]
    .median()
    .sort_values()
    .index.tolist()
)
box_data = [
    largest_lengths.loc[
        largest_lengths["language"] == language, "custom_bpe_token_count"
    ].to_numpy()
    for language in language_order
]

fig, ax = plt.subplots(figsize=(7.2, 5.0))
boxplot = ax.boxplot(
    box_data,
    orientation="horizontal",
    tick_labels=language_order,
    widths=0.62,
    patch_artist=True,
    showfliers=False,
    medianprops={"color": "#202020", "linewidth": 1.3},
    whiskerprops={"color": "#555555", "linewidth": 0.9},
    capprops={"color": "#555555", "linewidth": 0.9},
)
for box, language in zip(boxplot["boxes"], language_order):
    box.set_facecolor(SCRIPT_COLORS[language_to_family[language]])
    box.set_alpha(0.78)
    box.set_edgecolor("#333333")
    box.set_linewidth(0.8)

means = [values.mean() for values in box_data]
ax.scatter(
    means, np.arange(1, len(language_order) + 1), marker="D", s=20,
    facecolor="white", edgecolor="#202020", linewidth=0.8, zorder=3,
)
ax.set_title(f"BPE sequence-length distributions at {largest_checkpoint // 1000}k merges", loc="left", fontweight="bold")
ax.set_xlabel("Custom BPE sequence length (tokens)")
ax.set_ylabel("")
ax.grid(axis="x", color="#D9D9D9", linewidth=0.6, alpha=0.8)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
legend_handles = [
    Patch(facecolor=color, edgecolor="#333333", alpha=0.78, label=script)
    for script, color in SCRIPT_COLORS.items()
]
legend_handles.append(
    plt.Line2D([], [], marker="D", linestyle="None", markersize=5,
               markerfacecolor="white", markeredgecolor="#202020", label="Mean")
)
ax.legend(handles=legend_handles, title="Writing system", frameon=False, ncol=2, loc="lower right")
finish_figure(fig, "bpe_token_length_distributions_30k")

In [ ]:
# Figure 3: aligned comparison at the largest checkpoint.
heatmap_language_order = [
    metadata["language"]
    for family in ("Latin", "Cyrillic", "Indic")
    for metadata in SELECTED_LANGUAGES.values()
    if script_family(metadata["script"]) == family
]
heatmap_data = first_ten_custom_lengths.loc[largest_checkpoint, heatmap_language_order].T
values = heatmap_data.to_numpy()

fig, ax = plt.subplots(figsize=(8.2, 5.0))
image = ax.imshow(values, cmap="viridis", aspect="auto", interpolation="nearest")
ax.set_title(f"Aligned FLORES-101 sentence lengths at {largest_checkpoint // 1000}k merges", loc="left", fontweight="bold")
ax.set_xlabel("Aligned sentence ID")
ax.set_ylabel("")
ax.set_xticks(np.arange(values.shape[1]), labels=heatmap_data.columns.astype(str))
ax.set_yticks(np.arange(values.shape[0]), labels=heatmap_data.index)

threshold = (values.min() + values.max()) / 2
for row in range(values.shape[0]):
    for column in range(values.shape[1]):
        ax.text(column, row, f"{values[row, column]:.0f}", ha="center", va="center",
                fontsize=7.5, color="white" if values[row, column] < threshold else "#111111")

# Thin separators retain script groupings without adding visual clutter.
ax.axhline(3.5, color="white", linewidth=1.5)
ax.axhline(6.5, color="white", linewidth=1.5)
colorbar = fig.colorbar(image, ax=ax, pad=0.02, fraction=0.04)
colorbar.set_label("Custom BPE sequence length (tokens)")
for spine in ax.spines.values():
    spine.set_visible(False)
finish_figure(fig, "bpe_aligned_sentence_heatmap_30k")